In [0]:
%sql
SHOW TABLES IN workspace.supermarket;

In [0]:
%sql
SELECT
    s.store_id,
    s.store_name,
    ROUND(SUM(st.total_amount), 2) AS total_revenue
FROM workspace.supermarket.sales_transactions st
JOIN workspace.supermarket.stores s
    ON st.store_id = s.store_id
GROUP BY
    s.store_id,
    s.store_name
ORDER BY total_revenue DESC;

In [0]:
%sql
SELECT
    p.product_id,
    p.product_name,
    p.category,
    SUM(st.quantity_sold) AS total_quantity_sold
FROM workspace.supermarket.sales_transactions st
JOIN workspace.supermarket.products p
    ON st.product_id = p.product_id
GROUP BY
    p.product_id,
    p.product_name,
    p.category
ORDER BY total_quantity_sold DESC
LIMIT 5;

In [0]:
%sql
SELECT
    s.store_id,
    s.store_name,
    ROUND(SUM(st.total_amount), 2) AS total_revenue,

    RANK() OVER (
        ORDER BY SUM(st.total_amount) DESC
    ) AS revenue_rank

FROM workspace.supermarket.sales_transactions st

JOIN workspace.supermarket.stores s
    ON st.store_id = s.store_id

GROUP BY
    s.store_id,
    s.store_name

ORDER BY revenue_rank;

In [0]:
%sql
SELECT
    s.store_id,
    s.store_name,

    COUNT(DISTINCT st.transaction_id) AS transaction_count,

    RANK() OVER (
        ORDER BY COUNT(DISTINCT st.transaction_id) DESC
    ) AS transaction_rank

FROM workspace.supermarket.sales_transactions st

JOIN workspace.supermarket.stores s
    ON st.store_id = s.store_id

GROUP BY
    s.store_id,
    s.store_name

ORDER BY transaction_rank;

In [0]:
%sql
SELECT
    s.store_name,
    p.product_name,
    p.category,

    ROUND(
        SUM(st.total_amount),
        2
    ) AS product_revenue,

    RANK() OVER (
        PARTITION BY s.store_id
        ORDER BY SUM(st.total_amount) DESC
    ) AS product_rank

FROM workspace.supermarket.sales_transactions st

JOIN workspace.supermarket.stores s
    ON st.store_id = s.store_id

JOIN workspace.supermarket.products p
    ON st.product_id = p.product_id

GROUP BY
    s.store_id,
    s.store_name,
    p.product_name,
    p.category

ORDER BY
    s.store_id,
    product_rank;

In [0]:
%sql
SELECT
    DATE_FORMAT(sale_date, 'yyyy-MM') AS month,

    ROUND(
        SUM(total_amount),
        2
    ) AS monthly_revenue,

    SUM(quantity_sold) AS units_sold,

    COUNT(DISTINCT transaction_id) AS transactions

FROM workspace.supermarket.sales_transactions

GROUP BY
    DATE_FORMAT(sale_date, 'yyyy-MM')

ORDER BY month;

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
SELECT
    s.store_id,
    s.store_name,

    COUNT(
        DISTINCT st.product_id
    ) AS unique_products_sold

FROM workspace.supermarket.sales_transactions st

JOIN workspace.supermarket.stores s
    ON st.store_id = s.store_id

GROUP BY
    s.store_id,
    s.store_name

ORDER BY unique_products_sold DESC;

In [0]:
%sql
WITH store_metrics AS (

    SELECT
        s.store_id,
        s.store_name,

        SUM(st.total_amount) AS total_revenue,

        COUNT(DISTINCT st.transaction_id)
            AS transaction_count,

        COUNT(DISTINCT st.product_id)
            AS unique_products,

        SUM(st.quantity_sold)
            AS total_units_sold

    FROM workspace.supermarket.sales_transactions st

    JOIN workspace.supermarket.stores s
        ON st.store_id = s.store_id

    GROUP BY
        s.store_id,
        s.store_name
)

SELECT
    store_id,
    store_name,

    ROUND(total_revenue, 2)
        AS total_revenue,

    transaction_count,

    ROUND(
        total_revenue /
        NULLIF(transaction_count, 0),
        2
    ) AS average_transaction_value,

    unique_products,

    total_units_sold,

    RANK() OVER (
        ORDER BY total_revenue DESC
    ) AS revenue_rank,

    RANK() OVER (
        ORDER BY transaction_count DESC
    ) AS transaction_rank

FROM store_metrics

ORDER BY revenue_rank;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.supermarket.store_sales_summary
USING DELTA
AS

WITH store_metrics AS (

    SELECT
        s.store_id,
        s.store_name,

        SUM(st.total_amount) AS total_revenue,

        COUNT(DISTINCT st.transaction_id)
            AS transaction_count,

        COUNT(DISTINCT st.product_id)
            AS unique_products,

        SUM(st.quantity_sold)
            AS total_units_sold

    FROM workspace.supermarket.sales_transactions st

    JOIN workspace.supermarket.stores s
        ON st.store_id = s.store_id

    GROUP BY
        s.store_id,
        s.store_name
)

SELECT
    store_id,
    store_name,

    ROUND(total_revenue, 2)
        AS total_revenue,

    transaction_count,

    ROUND(
        total_revenue /
        NULLIF(transaction_count, 0),
        2
    ) AS average_transaction_value,

    unique_products,

    total_units_sold,

    RANK() OVER (
        ORDER BY total_revenue DESC
    ) AS revenue_rank,

    RANK() OVER (
        ORDER BY transaction_count DESC
    ) AS transaction_rank

FROM store_metrics;

In [0]:
%sql
SELECT *
FROM workspace.supermarket.store_sales_summary
ORDER BY revenue_rank;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.supermarket.monthly_sales_summary
USING DELTA
AS

SELECT
    DATE_FORMAT(sale_date, 'yyyy-MM') AS month,

    ROUND(
        SUM(total_amount),
        2
    ) AS monthly_revenue,

    SUM(quantity_sold) AS units_sold,

    COUNT(DISTINCT transaction_id)
        AS transaction_count

FROM workspace.supermarket.sales_transactions

GROUP BY
    DATE_FORMAT(sale_date, 'yyyy-MM');

In [0]:
%sql
SELECT *
FROM workspace.supermarket.monthly_sales_summary
ORDER BY month;

In [0]:
%sql
SHOW TABLES IN workspace.supermarket;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.supermarket.product_sales_summary
USING DELTA
AS

SELECT
    p.product_id,
    p.product_name,
    p.category,

    SUM(st.quantity_sold)
        AS total_quantity_sold,

    ROUND(
        SUM(st.total_amount),
        2
    ) AS total_revenue,

    COUNT(DISTINCT st.store_id)
        AS stores_selling_product,

    COUNT(DISTINCT st.transaction_id)
        AS transaction_count

FROM workspace.supermarket.sales_transactions st

JOIN workspace.supermarket.products p
    ON st.product_id = p.product_id

GROUP BY
    p.product_id,
    p.product_name,
    p.category;

In [0]:
%sql
SELECT *
FROM workspace.supermarket.product_sales_summary
ORDER BY total_quantity_sold DESC
LIMIT 10;